In [1]:
%pip install lightgbm catboost scikit-learn pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\rupes\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import zipfile
import os
import numpy as np

print("="*80)
print("ECOGRID AI: TEMPORALLY STRICT DATASET INTEGRATION PIPELINE")
print("="*80)

# 1. Load Local UCI Occupancy Dataset
print("\n[STEP 1/3] Loading UCI Occupancy Detection dataset...")
second_dataset_path = "occupancy+detection.zip"
extract_dir = "occupancy_extracted"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(second_dataset_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

occupancy_train = pd.read_csv(os.path.join(extract_dir, 'datatraining.txt'))
occupancy_test1 = pd.read_csv(os.path.join(extract_dir, 'datatest.txt'))
occupancy_test2 = pd.read_csv(os.path.join(extract_dir, 'datatest2.txt'))

occupancy_combined = pd.concat([occupancy_train, occupancy_test1, occupancy_test2], ignore_index=True)

# 2. Sort Chronologically
occupancy_combined['date'] = pd.to_datetime(occupancy_combined['date'])
occupancy_combined = occupancy_combined.sort_values('date').reset_index(drop=True)
occupancy_combined['hourly_timestamp'] = occupancy_combined['date'].dt.floor('h')

# Aggregate ground-truth occupancy ratio
occupancy_features = occupancy_combined.groupby('hourly_timestamp').agg({
    'Temperature': 'mean',
    'Humidity': 'mean',
    'Light': 'mean',
    'CO2': 'mean',
    'HumidityRatio': 'mean',
    'Occupancy': ['mean', 'max']
}).reset_index()

# Flatten columns
occupancy_features.columns = ['hourly_timestamp', 'Ambient_Temp_C', 'Humidity', 'Light', 'CO2_Level', 'HumidityRatio', 'Occupancy_Ratio', 'Occupancy_State_Binary']
occupancy_features = occupancy_features.sort_values('hourly_timestamp').reset_index(drop=True)

occupancy_features['Hour'] = occupancy_features['hourly_timestamp'].dt.hour
occupancy_features['DayOfWeek'] = occupancy_features['hourly_timestamp'].dt.dayofweek
occupancy_features['IsWeekend'] = (occupancy_features['DayOfWeek'] >= 5).astype(int)

# Cyclical time transformation
occupancy_features['Hour_Sin'] = np.sin(2 * np.pi * occupancy_features['Hour'] / 24.0)
occupancy_features['Hour_Cos'] = np.cos(2 * np.pi * occupancy_features['Hour'] / 24.0)

# Ground-truth categorical mapping
def map_occupancy_category_leak_free(row):
    ratio = row['Occupancy_Ratio']
    if ratio == 0.0:
        return 'Low'
    elif ratio < 0.5:
        return 'Medium'
    else:
        return 'High'

occupancy_features['Occupancy_Category'] = occupancy_features.apply(map_occupancy_category_leak_free, axis=1)

# HVAC Load Simulation with realistic stochastic disturbance
np.random.seed(42)
occupancy_weights = {"High": 25.0, "Medium": 12.0, "Low": 2.0}

def generate_hvac_load(r):
    base_load = 10.0
    occ_load = occupancy_weights[r["Occupancy_Category"]]
    temp_delta = max(0, (r['Ambient_Temp_C'] - 20.0) * 1.5)
    latent_variance = (r['Humidity'] / 100.0) * 3.0 + np.random.normal(0, 1.5)
    return round(base_load + occ_load + temp_delta + latent_variance, 2)

occupancy_features['HVAC_Power_kW'] = occupancy_features.apply(generate_hvac_load, axis=1)

occupancy_features.to_csv('ecogrid_integrated_matrix.csv', index=False)
print("="*80)
print("✅ ECOGRID TEMPORAL FEATURE MATRIX SAVED: ecogrid_integrated_matrix.csv")
print("="*80)

ECOGRID AI: TEMPORALLY STRICT DATASET INTEGRATION PIPELINE

[STEP 1/3] Loading UCI Occupancy Detection dataset...
✅ ECOGRID TEMPORAL FEATURE MATRIX SAVED: ecogrid_integrated_matrix.csv


In [3]:
# =====================================================================
# ECOGRID AI: STRICT TEMPORAL VALIDATION PIPELINE (ZERO LEAKAGE)
# =====================================================================

import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

def console_log(msg: str, delay: float = 0.01):
    print(msg, flush=True)
    time.sleep(delay)

class EcoGridEngine:
    def __init__(self):
        self.label_encoder = LabelEncoder()
        self.feature_cols = [
            'Hour_Sin', 'Hour_Cos', 'DayOfWeek', 'IsWeekend', 
            'Ambient_Temp_C', 'Temp_Rolling_Mean', 
            'CO2_Level', 'CO2_Rolling_Mean', 
            'Light', 'Light_Rolling_Mean', 'Humidity'
        ]

        self.lgb_params_cls = {'n_estimators': 40, 'max_depth': 3, 'num_leaves': 7, 'learning_rate': 0.03, 'reg_lambda': 3.0, 'random_state': 42, 'verbose': -1}
        self.cat_params_cls = {'iterations': 40, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0, 'random_state': 42, 'verbose': 0}
        
        self.lgb_params_reg = {'n_estimators': 40, 'max_depth': 3, 'num_leaves': 7, 'learning_rate': 0.03, 'reg_lambda': 3.0, 'random_state': 42, 'verbose': -1}
        self.cat_params_reg = {'iterations': 40, 'depth': 3, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0, 'random_state': 42, 'verbose': 0}

    def run_pipeline(self):
        console_log("="*80)
        console_log(" 🚀 ECOGRID AI: MULTI-TASK EVALUATION (STRICT TEMPORAL CUTOFF)")
        console_log("="*80)

        df = pd.read_csv('ecogrid_integrated_matrix.csv')
        df['hourly_timestamp'] = pd.to_datetime(df['hourly_timestamp'])
        df = df.sort_values('hourly_timestamp').reset_index(drop=True)

        # 1. TEMPORAL CUTOFF SPLIT (80% Historical Train | 20% Unseen Future Test)
        split_idx = int(len(df) * 0.8)
        train_df = df.iloc[:split_idx].copy()
        test_df = df.iloc[split_idx:].copy()

        # 2. LEAK-FREE ROLLING COMPUTATION (Calculated independently per split)
        for d in [train_df, test_df]:
            d['Temp_Rolling_Mean'] = d['Ambient_Temp_C'].rolling(window=3, min_periods=1).mean()
            d['CO2_Rolling_Mean'] = d['CO2_Level'].rolling(window=3, min_periods=1).mean()
            d['Light_Rolling_Mean'] = d['Light'].rolling(window=3, min_periods=1).mean()

        # Label encoding trained ONLY on historical training labels
        train_df['Occupancy_Label'] = self.label_encoder.fit_transform(train_df['Occupancy_Category'])
        test_df['Occupancy_Label'] = self.label_encoder.transform(test_df['Occupancy_Category'])

        X_train, y_train_cls, y_train_reg = train_df[self.feature_cols], train_df["Occupancy_Label"], train_df["HVAC_Power_kW"]
        X_test, y_test_cls, y_test_reg = test_df[self.feature_cols], test_df["Occupancy_Label"], test_df["HVAC_Power_kW"]

        console_log(f" ↳ Temporal Train Horizon: {train_df['hourly_timestamp'].min()} to {train_df['hourly_timestamp'].max()}")
        console_log(f" ↳ Temporal Test Horizon : {test_df['hourly_timestamp'].min()} to {test_df['hourly_timestamp'].max()}")

        # Fit models
        m1_cls = lgb.LGBMClassifier(**self.lgb_params_cls).fit(X_train, y_train_cls)
        m2_cls = CatBoostClassifier(**self.cat_params_cls).fit(X_train, y_train_cls)

        m1_reg = lgb.LGBMRegressor(**self.lgb_params_reg).fit(X_train, y_train_reg)
        m2_reg = CatBoostRegressor(**self.cat_params_reg).fit(X_train, y_train_reg)

        # Evaluate metrics
        tr_prob = (m1_cls.predict_proba(X_train) + m2_cls.predict_proba(X_train)) / 2
        te_prob = (m1_cls.predict_proba(X_test) + m2_cls.predict_proba(X_test)) / 2
        train_acc = accuracy_score(y_train_cls, np.argmax(tr_prob, axis=1))
        test_acc = accuracy_score(y_test_cls, np.argmax(te_prob, axis=1))

        tr_pred_reg = (m1_reg.predict(X_train) + m2_reg.predict(X_train)) / 2
        te_pred_reg = (m1_reg.predict(X_test) + m2_pred_reg) / 2 if 'm2_pred_reg' in locals() else (m1_reg.predict(X_test) + m2_reg.predict(X_test)) / 2
        train_rmse = np.sqrt(mean_squared_error(y_train_reg, tr_pred_reg))
        test_rmse = np.sqrt(mean_squared_error(y_test_reg, te_pred_reg))

        console_log(f"\n 📑 Classification Accuracy -> Train: {train_acc*100:.1f}% | Test: {test_acc*100:.1f}%")
        console_log(f" 📑 Forecasting RMSE       -> Train: {train_rmse:.2f} kW | Test: {test_rmse:.2f} kW")
        console_log("="*80)

if __name__ == "__main__":
    engine = EcoGridEngine()
    engine.run_pipeline()

 🚀 ECOGRID AI: MULTI-TASK EVALUATION (STRICT TEMPORAL CUTOFF)
 ↳ Temporal Train Horizon: 2015-02-02 14:00:00 to 2015-02-15 11:00:00
 ↳ Temporal Test Horizon : 2015-02-15 12:00:00 to 2015-02-18 09:00:00

 📑 Classification Accuracy -> Train: 95.7% | Test: 97.1%
 📑 Forecasting RMSE       -> Train: 4.34 kW | Test: 4.35 kW
